# Voice Interview Simulator

**Powered by Mistral Voxtral**

---

This notebook simulates a realistic AI-driven job interview using:

| Component | Model / Library |
|---|---|
| Resume parsing | Reuses `pipeline/ocr.py` from this project |
| Question generation | `mistral-large-latest` (high-reasoning) |
| Answer transcription (STT) | `voxtral-mini-latest` via `client.audio.transcriptions.complete` |
| Answer understanding & evaluation | `voxtral-small-latest` (chat with audio) |
| Interviewer voice (TTS) | `pyttsx3` — offline, cross-platform, no extra API key |
| Microphone capture | `sounddevice` + `scipy` |

### How it works

```
Resume (PDF/JSON)
      │
      ▼
  OCR + LLM parse  ──►  Structured resume dict
                                │
                                ▼
              mistral-large-latest generates N questions
                                │
                    ┌───────────┴───────────┐
                    │   Interview Loop      │
                    │  1. TTS speaks Q      │
                    │  2. Mic records ans   │
                    │  3. Voxtral STT       │
                    │  4. LLM evaluates     │
                    │  5. TTS speaks FB     │
                    └───────────────────────┘
                                │
                                ▼
                      Final Interview Report
```

### Prerequisites

Install dependencies once:

```bash
pip install python-dotenv mistralai sounddevice scipy pyttsx3 pydub ipython
```

Create / update your `.env` file in the project root:

```
MISTRAL_API_KEY=your_key_here
```

---
## Cell 1 — Install Dependencies

Run once. Safe to skip if already installed.

In [3]:
# Install required packages (run once)
import subprocess, sys

packages = [
    "python-dotenv",
    "mistralai",
    "sounddevice",
    "scipy",       # WAV file writing (no external binary needed)
    "pyttsx3",     # Offline TTS — works on Windows / macOS / Linux
    "pydub",       # Audio format helpers
    "ipython",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])

print("All dependencies installed.")

All dependencies installed.


---
## Cell 2 — Environment Configuration

Loads `MISTRAL_API_KEY` from the `.env` file in the project root.  
The key is **never** hardcoded.

In [7]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# ── Locate project root (the notebook lives at project root)
PROJECT_ROOT = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').resolve().parent
# Fallback: assume the notebook is in the project root
PROJECT_ROOT = Path(".").resolve()

# # ── Load .env
# env_path = PROJECT_ROOT / ".env"
# if not env_path.exists():
#     raise FileNotFoundError(
#         f".env file not found at {env_path}\n"
#         "Create it with: MISTRAL_API_KEY=your_key_here"
#     )

# load_dotenv(env_path)
API_KEY = "kNVCjm7F0xiaTEcDQIKVQmkXhVvjYmSX"

if not API_KEY:
    raise ValueError("MISTRAL_API_KEY is empty. Check your .env file.")

# ── Add project root to sys.path so pipeline/ imports work
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"API key      : {API_KEY[:8]}{'*' * (len(API_KEY) - 8)}")

Project root : D:\ilyass\ptorjcts\Career_assitant\agents\test
API key      : kNVCjm7F************************


---
## Cell 3 — Mistral Client Initialisation

A single shared client is created here and reused by all subsequent cells.

In [8]:
from mistralai import Mistral

client = Mistral(api_key=API_KEY)

# Model constants
QUESTION_MODEL    = "mistral-large-latest"     # High-reasoning — generates interview questions
EVALUATION_MODEL  = "mistral-large-latest"     # High-reasoning — evaluates answers
STT_MODEL         = "voxtral-mini-latest"      # Voxtral STT — transcribes microphone audio
AUDIO_CHAT_MODEL  = "voxtral-small-latest"     # Voxtral chat-with-audio — understands spoken answers

print(f"Question/Evaluation model : {QUESTION_MODEL}")
print(f"STT model                 : {STT_MODEL}")
print(f"Audio chat model          : {AUDIO_CHAT_MODEL}")

Question/Evaluation model : mistral-large-latest
STT model                 : voxtral-mini-latest
Audio chat model          : voxtral-small-latest


---
## Cell 4 — Resume Loading

Two modes:

1. **From a saved JSON file** — if you already ran the Career Assistant web app and have a `strategist_output.json`, point `RESUME_SOURCE` to it.
2. **From a PDF / image** — the existing `pipeline/ocr.py` pipeline will OCR and parse it in real time.

Set `RESUME_SOURCE` below.

In [9]:
import json

# ─────────────────────────────────────────────────────────────
# CONFIGURE THIS — point to your resume
# ─────────────────────────────────────────────────────────────
# Option A: path to an already-parsed JSON resume
#   (e.g. agents/test/resume_llm_structured.json)
# Option B: path to a PDF or image resume
#   (e.g. uploads/abc123/resume.pdf)
# Leave as None to use the bundled sample JSON.

RESUME_SOURCE = r"C:\Users\HP\Documents\Career_assitant\agents\test\resume_llm_structured.json"   # str | None

# Optional: target job role for context-aware questions
TARGET_ROLE    = "Senior Software Engineer"   # e.g. "Data Scientist", "ML Engineer"
TARGET_COMPANY = "a top-tier tech company"   # e.g. "Google", "OpenAI"
# ─────────────────────────────────────────────────────────────


def load_resume(source: str | None) -> dict:
    """Load a resume from a JSON file, a PDF/image path, or fall back to sample."""

    # ── Fallback: bundled sample resume
    if source is None:
        sample_path = PROJECT_ROOT / "agents" / "test" / "resume_llm_structured.json"
        if sample_path.exists():
            print(f"No RESUME_SOURCE set — using bundled sample: {sample_path.name}")
            return json.loads(sample_path.read_text(encoding="utf-8"))
        else:
            raise FileNotFoundError(
                "No RESUME_SOURCE set and bundled sample not found.\n"
                "Set RESUME_SOURCE to a .json or .pdf/.png/.jpg file."
            )

    source_path = Path(source)
    if not source_path.exists():
        raise FileNotFoundError(f"Resume file not found: {source_path}")

    ext = source_path.suffix.lower()

    # ── JSON: load directly
    if ext == ".json":
        print(f"Loading parsed resume from JSON: {source_path.name}")
        return json.loads(source_path.read_text(encoding="utf-8"))

    # ── PDF / image: run the existing OCR pipeline
    if ext in (".pdf", ".png", ".jpg", ".jpeg"):
        print(f"Running OCR pipeline on: {source_path.name} ...")
        from pipeline.ocr import run_ocr_pipeline
        return run_ocr_pipeline(str(source_path), progress_cb=print)

    raise ValueError(f"Unsupported resume format: {ext}")


resume = load_resume(RESUME_SOURCE)

# ── Pretty-print key fields
name     = resume.get("name", "Unknown Candidate")
sections = resume.get("sections", {})
skills   = sections.get("skills", [])
exp      = sections.get("experience", [])
projects = sections.get("projects", [])
edu      = sections.get("education", [])

print(f"\nCandidate   : {name}")
print(f"Skills      : {', '.join(skills[:8])}{'...' if len(skills) > 8 else ''}")
print(f"Experience  : {len(exp)} role(s)")
print(f"Projects    : {len(projects)} project(s)")
print(f"Education   : {len(edu)} entry(s)")

FileNotFoundError: Resume file not found: C:\Users\HP\Documents\Career_assitant\agents\test\resume_llm_structured.json

---
## Cell 5 — Resume Context Builder

Formats the parsed resume into a concise plain-text block that is injected into LLM prompts.  
This keeps every downstream prompt grounded in the candidate's actual background.

In [ ]:
def build_resume_context(resume: dict) -> str:
    """
    Convert the structured resume dict into a terse, LLM-friendly text block.
    Only factual information from the resume is included — nothing is fabricated.
    """
    s = resume.get("sections", {})
    lines = []

    lines.append(f"CANDIDATE: {resume.get('name', 'Unknown')}")

    contact = resume.get("contact", {})
    if contact.get("location"):
        lines.append(f"LOCATION: {contact['location']}")

    if s.get("summary"):
        lines.append(f"\nSUMMARY:\n{s['summary']}")

    # Skills
    skills = s.get("skills", [])
    if skills:
        lines.append(f"\nSKILLS: {', '.join(skills)}")

    # Experience
    experience = s.get("experience", [])
    if experience:
        lines.append("\nWORK EXPERIENCE:")
        for role in experience:
            title = role.get("title", "")
            org   = role.get("organisation", "")
            start = role.get("start_date", "")
            end   = role.get("end_date", "")
            desc  = role.get("description", "")
            bullets = role.get("bullets", [])
            lines.append(f"  • {title} at {org} ({start} – {end})")
            if desc:
                lines.append(f"    {desc}")
            for b in bullets:
                lines.append(f"    - {b}")

    # Projects
    projects = s.get("projects", [])
    if projects:
        lines.append("\nPROJECTS:")
        for p in projects:
            title = p.get("title", "")
            techs = ", ".join(p.get("technologies", []))
            desc  = p.get("description", "")
            lines.append(f"  • {title}" + (f" [{techs}]" if techs else ""))
            if desc:
                lines.append(f"    {desc}")

    # Education
    education = s.get("education", [])
    if education:
        lines.append("\nEDUCATION:")
        for e in education:
            lines.append(f"  • {e.get('title','')} — {e.get('organisation','')} ({e.get('end_date','')})")
            if e.get("description"):
                lines.append(f"    {e['description']}")

    # Certificates
    certs = s.get("certificates", [])
    if certs:
        lines.append("\nCERTIFICATES:")
        for c in certs:
            lines.append(f"  • {c.get('title','')} — {c.get('organisation','')}")

    return "\n".join(lines)


RESUME_CONTEXT = build_resume_context(resume)
print(RESUME_CONTEXT)

CANDIDATE: John Doe
LOCATION: New York, USA

SUMMARY:
Highly motivated and experienced Data Scientist with a strong background in machine learning, data analysis, and database administration. Proven track record of delivering high-quality projects and driving business growth through data-driven insights.

SKILLS: Machine Learning, Data Analysis, Database Administration, Python, R, SQL, Apache Spark, Communication, Team Management

WORK EXPERIENCE:
  • Senior Data Scientist at ABC Corporation (Jan 2020 – Present)
    Led a team of data scientists to develop and implement predictive models that increased sales by 25% and reduced costs by 15%. Collaborated with cross-functional teams to design and deploy data pipelines, resulting in a 30% improvement in data quality and a 40% reduction in data processing time.
    - Developed and deployed machine learning models using Python, R, and SQL
    - Designed and implemented data pipelines using Apache Beam, Apache Spark, and AWS Glue
    - Colla

---
## Cell 6 — Interview Question Generation

Uses `mistral-large-latest` to generate a personalised set of interview questions  
across four categories based on the candidate's actual resume.

In [ ]:
def generate_interview_questions(
    resume_context: str,
    target_role: str,
    target_company: str,
    num_questions: int = 7,
) -> list[dict]:
    """
    Generate structured interview questions using mistral-large-latest.

    Returns a list of dicts:
        [
          {"category": "Technical", "question": "..."},
          {"category": "Behavioral", "question": "..."},
          ...
        ]
    """
    system_prompt = (
        "You are a senior technical interviewer at a top-tier technology company. "
        "Your role is to conduct rigorous, fair, and insightful interviews. "
        "You craft questions that reveal genuine depth of knowledge and real-world experience. "
        "You never ask generic questions — every question is grounded in the candidate's specific background."
    )

    user_prompt = f"""You are interviewing the following candidate for the role of {target_role} at {target_company}.

=== CANDIDATE RESUME ===
{resume_context}
========================

Generate exactly {num_questions} interview questions personalised to this candidate.
Distribute them across these categories:
  1. Technical      — test depth of knowledge in their stated skills and tech stack
  2. Behavioral     — use STAR method, reference their actual experience
  3. Project deep dive — probe a specific project or achievement from their resume
  4. Problem solving  — a realistic scenario relevant to the target role

Rules:
  - Every question must reference something specific from the resume (skill, project, company, or achievement)
  - Questions must be clear and speakable (no bullet sub-points, no markdown)
  - Vary difficulty: mix moderate and challenging questions
  - Do NOT include the answer

Return ONLY a valid JSON array in this exact format (no markdown fences, no extra text):
[
  {{"category": "Technical", "question": "..."}},
  {{"category": "Behavioral", "question": "..."}},
  ...
]"""

    response = client.chat.complete(
        model=QUESTION_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.7,
        max_tokens=2048,
    )

    raw = response.choices[0].message.content.strip()

    # Strip any accidental markdown fences
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    return json.loads(raw)


# ── Generate questions
print(f"Generating interview questions for '{TARGET_ROLE}' at '{TARGET_COMPANY}' ...\n")
QUESTIONS = generate_interview_questions(
    RESUME_CONTEXT,
    target_role=TARGET_ROLE,
    target_company=TARGET_COMPANY,
    num_questions=7,
)

print(f"Generated {len(QUESTIONS)} questions:\n")
for i, q in enumerate(QUESTIONS, 1):
    print(f"  Q{i} [{q['category']}]")
    print(f"     {q['question']}")
    print()

Generating interview questions for 'Senior Software Engineer' at 'a top-tier tech company' ...

Generated 7 questions:

  Q1 [Technical]
     At ABC Corporation, you used Apache Spark and AWS Glue to design data pipelines. Can you walk me through a scenario where you had to optimize a Spark job that was running slower than expected? What specific techniques or configurations did you use to improve its performance?

  Q2 [Technical]
     You’ve listed both Python and R as languages you’ve used for machine learning. In your experience, what are the key trade-offs between using Python’s scikit-learn and R’s caret or tidymodels for building predictive models? Can you give an example where you chose one over the other and why?

  Q3 [Behavioral]
     At DEF Startups, you mentioned collaborating with data engineers to design data pipelines. Tell me about a time when there was a misalignment between your team and the data engineers on a project. How did you identify the issue, and what steps 

---
## Cell 7 — Text-to-Speech (Interviewer Voice)

`pyttsx3` runs fully offline — no API key, no network call, no latency.  
It uses the OS speech engine (SAPI5 on Windows, NSSpeechSynthesizer on macOS, espeak on Linux).

In [ ]:
import pyttsx3
import threading

# Initialise TTS engine once
_tts_engine = pyttsx3.init()
_tts_engine.setProperty("rate",   165)   # words per minute (default ~200, slower = clearer)
_tts_engine.setProperty("volume", 1.0)  # Max volume

# Try to select a higher-quality voice if available
_voices = _tts_engine.getProperty("voices")
# Prefer a voice that sounds professional — pick the second voice if available
if len(_voices) > 1:
    _tts_engine.setProperty("voice", _voices[1].id)

# Thread lock — pyttsx3 is not thread-safe
_tts_lock = threading.Lock()


def speak(text: str, label: str = "") -> None:
    """
    Speak `text` aloud using the OS TTS engine.
    Blocks until the utterance is complete.
    Also prints the text so the user can read along.
    """
    global _tts_engine
    
    if label:
        print(f"\n[{label}] {text}")
    else:
        print(f"\n{text}")

    try:
        with _tts_lock:
            _tts_engine.say(text)
            _tts_engine.runAndWait()
    except Exception as e:
        print(f"  [TTS Warning: {e}]")
        # Try to reinitialize and retry once
        try:
            print("  [Reinitializing TTS engine...]")
            _tts_engine.stop()
            _tts_engine = pyttsx3.init()
            _tts_engine.setProperty("rate", 165)
            _tts_engine.setProperty("volume", 1.0)
            if len(_voices) > 1:
                _tts_engine.setProperty("voice", _voices[1].id)
            _tts_engine.say(text)
            _tts_engine.runAndWait()
            print("  [TTS recovered]")
        except Exception as e2:
            print(f"  [TTS Error: Could not speak. {e2}]")


# Quick test
speak("Text to speech engine initialised successfully.", label="TTS TEST")


[TTS TEST] Text to speech engine initialised successfully.


---
## 🔧 TROUBLESHOOTING — Test TTS Audio

If you don't hear the interviewer speaking, run the cell below to diagnose the issue.

In [ ]:
# TTS Audio Diagnostic — Run this if you can't hear the interviewer
import pyttsx3

print("Testing Text-to-Speech...")

# Check if TTS engine is already initialized
try:
    _tts_engine
    print("✓ TTS engine already initialized from Cell 7")
except NameError:
    print("⚠ TTS engine not initialized yet. Initializing now...")
    _tts_engine = pyttsx3.init()

# Get available voices
_voices = _tts_engine.getProperty("voices")
print(f"Available voices: {len(_voices)}")

# List available voices
for i, voice in enumerate(_voices):
    print(f"  {i}: {voice.name}")

# Test current settings
current_voice = _tts_engine.getProperty("voice")
print(f"\nCurrent voice: {current_voice[:80] if current_voice else 'Not set'}")
print(f"Current rate : {_tts_engine.getProperty('rate')} words/min")
print(f"Current volume: {_tts_engine.getProperty('volume')}")

# Set to maximum volume and test
print("\n🔊 Testing speech at max volume...")
_tts_engine.setProperty("volume", 1.0)
_tts_engine.setProperty("rate", 150)

print("\n[TTS TEST] Testing audio now...")
_tts_engine.say("Testing. Can you hear me now?")
_tts_engine.runAndWait()

print("\n" + "="*60)
print("💡 TROUBLESHOOTING TIPS:")
print("="*60)
print("1. Check Windows volume (bottom-right taskbar)")
print("2. Check Volume Mixer - make sure Python/Jupyter isn't muted")
print("3. Verify correct audio output device is set as default")
print("\nIf you still don't hear anything, try a different voice:")
print("   _tts_engine.setProperty('voice', _voices[0].id)")
print("   _tts_engine.say('Testing voice zero')")
print("   _tts_engine.runAndWait()")

Testing Text-to-Speech...
✓ TTS engine already initialized from Cell 7
Available voices: 3
  0: Microsoft David Desktop - English (United States)
  1: Microsoft Zira Desktop - English (United States)
  2: Microsoft Hortense Desktop - French

Current voice: HKEY_LOCAL_MACHINE\SOFTWARE\Microsoft\Speech\Voices\Tokens\TTS_MS_EN-US_ZIRA_11.
Current rate : 165 words/min
Current volume: 1.0

🔊 Testing speech at max volume...

[TTS TEST] Testing audio now...

💡 TROUBLESHOOTING TIPS:
1. Check Windows volume (bottom-right taskbar)
2. Check Volume Mixer - make sure Python/Jupyter isn't muted
3. Verify correct audio output device is set as default

If you still don't hear anything, try a different voice:
   _tts_engine.setProperty('voice', _voices[0].id)
   _tts_engine.say('Testing voice zero')
   _tts_engine.runAndWait()


---
## Cell 8 — Microphone Audio Capture

Records from the default microphone for a fixed duration using `sounddevice`,  
saves the audio as a 16-bit WAV file using `scipy.io.wavfile`,  
and plays it back via `IPython.display.Audio` for confirmation.

In [ ]:
import tempfile
import time
import numpy as np
import sounddevice as sd
from scipy.io import wavfile
from IPython.display import Audio, display

# Audio recording settings
SAMPLE_RATE   = 16000   # Hz — Voxtral STT works well at 16 kHz
CHANNELS      = 1       # Mono
MAX_DURATION  = 90      # seconds — safety cap per answer


def record_answer(
    duration_seconds: int = 60,
    sample_rate: int = SAMPLE_RATE,
    silence_threshold: float = 0.003,
    silence_timeout: float = 3.0,
) -> str:
    """
    Record microphone audio until either:
      a) `duration_seconds` elapsed, or
      b) `silence_timeout` seconds of continuous silence detected

    Returns the path to a temporary WAV file.

    Parameters
    ----------
    duration_seconds  : Maximum recording time.
    sample_rate       : Audio sample rate in Hz.
    silence_threshold : RMS amplitude below which audio is considered silent.
    silence_timeout   : Stop recording after this many seconds of silence.
    """
    print(f"\n  Recording... (speak now, max {duration_seconds}s, auto-stops after {silence_timeout}s silence)")
    print("  Press Ctrl+C to stop early.")

    chunk_size    = int(sample_rate * 0.5)   # 500ms chunks for silence detection
    all_chunks    = []
    silent_chunks = 0
    silent_limit  = int(silence_timeout / 0.5)
    total_chunks  = int(duration_seconds  / 0.5)

    try:
        for _ in range(total_chunks):
            chunk = sd.rec(
                chunk_size,
                samplerate=sample_rate,
                channels=CHANNELS,
                dtype="float32",
            )
            sd.wait()
            all_chunks.append(chunk)

            rms = float(np.sqrt(np.mean(chunk ** 2)))
            if rms < silence_threshold:
                silent_chunks += 1
            else:
                silent_chunks = 0   # reset on speech

            if silent_chunks >= silent_limit and len(all_chunks) > silent_limit:
                print("  (silence detected — stopping)")
                break

    except KeyboardInterrupt:
        print("  (stopped by user)")

    if not all_chunks:
        raise RuntimeError("No audio was recorded.")

    # Concatenate and save to temp WAV
    audio_data = np.concatenate(all_chunks, axis=0)
    audio_int16 = (audio_data * 32767).astype(np.int16)

    tmp = tempfile.NamedTemporaryFile(
        suffix=".wav",
        delete=False,
        prefix="interview_answer_",
    )
    wavfile.write(tmp.name, sample_rate, audio_int16)
    tmp.close()

    duration = len(audio_data) / sample_rate
    print(f"  Recorded {duration:.1f}s → {Path(tmp.name).name}")

    return tmp.name


def playback_audio(wav_path: str) -> None:
    """Play back a WAV file inline in the notebook for confirmation."""
    print("  Playback:")
    display(Audio(filename=wav_path, autoplay=False))


print("Audio recording helpers ready.")
print(f"Default input device: {sd.query_devices(kind='input')['name']}")

Audio recording helpers ready.
Default input device: Réseau de microphones (Technolo


---
## Cell 9 — Speech-to-Text Transcription (Voxtral)

Sends the recorded WAV file to `voxtral-mini-latest` via  
`client.audio.transcriptions.complete()` and returns the transcript text.

In [ ]:
def transcribe_audio(wav_path: str, context_hints: list[str] | None = None) -> str:
    """
    Transcribe a WAV file using Mistral Voxtral STT.

    Parameters
    ----------
    wav_path      : Path to the WAV file recorded from the microphone.
    context_hints : Optional list of domain-specific terms/skills to improve
                    transcription accuracy (Voxtral context biasing).

    Returns
    -------
    str : Transcribed text.
    """
    print("  Transcribing with Voxtral...")

    kwargs = {
        "model": STT_MODEL,
        "language": "en",
    }

    # Inject skills as context bias for better technical term recognition
    if context_hints:
        # Voxtral accepts a comma-separated string of up to 100 terms
        bias = ",".join(context_hints[:100])
        kwargs["context_bias"] = bias

    with open(wav_path, "rb") as f:
        response = client.audio.transcriptions.complete(
            file={"content": f, "file_name": Path(wav_path).name},
            **kwargs,
        )

    transcript = response.text.strip()
    print(f"  Transcript: \"{transcript}\"")
    return transcript


# Build a skill-based context bias list from the candidate's resume
STT_CONTEXT_HINTS = (
    resume.get("sections", {}).get("skills", []) +
    [p.get("title", "") for p in resume.get("sections", {}).get("projects", [])]
)

print(f"STT context hints ({len(STT_CONTEXT_HINTS)} terms): {STT_CONTEXT_HINTS[:10]}")

STT context hints (11 terms): ['Machine Learning', 'Data Analysis', 'Database Administration', 'Python', 'R', 'SQL', 'Apache Spark', 'Communication', 'Team Management', 'Customer Segmentation']


---
## Cell 10 — Answer Evaluation

Uses `mistral-large-latest` to evaluate each transcribed answer against four dimensions:  
technical accuracy, communication clarity, confidence, and relevance to the resume.  
Returns a structured JSON score object.

Two evaluation paths are available:
- **Text-only** (always available): sends the transcript text to the reasoning model.
- **Audio-native** (Voxtral chat): sends the raw audio to `voxtral-small-latest` for
  evaluation that also captures prosody and tone.

In [ ]:
import base64


def evaluate_answer_text(
    question: str,
    category: str,
    transcript: str,
    resume_context: str,
) -> dict:
    """
    Evaluate a candidate's transcribed answer using mistral-large-latest.

    Returns a dict:
    {
        "technical_accuracy": int (0-100),
        "communication_clarity": int (0-100),
        "confidence": int (0-100),
        "relevance": int (0-100),
        "overall_score": int (0-100),
        "strengths": [str, ...],
        "improvements": [str, ...],
        "brief_feedback": str   <- 2-3 sentences, speakable
    }
    """
    system_prompt = (
        "You are an expert interview coach and technical interviewer. "
        "Evaluate candidate answers rigorously and constructively. "
        "Be honest — do not inflate scores. Base feedback on evidence in the answer."
    )

    user_prompt = f"""Evaluate this interview answer.

QUESTION ({category}): {question}

CANDIDATE ANSWER (transcribed from speech):
\"\"\"{transcript}\"\"\"

CANDIDATE BACKGROUND (for relevance scoring):
{resume_context[:1500]}  

Score the answer on each dimension (0-100):
  - technical_accuracy   : correctness and depth of technical content
  - communication_clarity: how clearly and concisely the answer was expressed
  - confidence           : how assured and decisive the response sounds
  - relevance            : how well the answer addresses the question using their background
  - overall_score        : weighted composite (technical 40%, clarity 25%, confidence 15%, relevance 20%)

Also provide:
  - strengths    : list of 1-3 specific positive points
  - improvements : list of 1-3 specific areas to improve
  - brief_feedback : 2-3 sentence spoken feedback (no bullet points, no markdown)

Return ONLY valid JSON (no markdown fences, no extra text):
{{
  "technical_accuracy": 0,
  "communication_clarity": 0,
  "confidence": 0,
  "relevance": 0,
  "overall_score": 0,
  "strengths": [],
  "improvements": [],
  "brief_feedback": ""
}}"""

    response = client.chat.complete(
        model=EVALUATION_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.3,
        max_tokens=1024,
    )

    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    return json.loads(raw)


def evaluate_answer_audio(
    question: str,
    category: str,
    wav_path: str,
    transcript: str,
) -> dict:
    """
    Evaluate using Voxtral small (audio-native chat).
    This also captures prosody, pacing, and filler words from the raw audio.
    Falls back to text-only evaluation if the audio call fails.
    """
    try:
        with open(wav_path, "rb") as f:
            audio_b64 = base64.b64encode(f.read()).decode("utf-8")

        prompt = (
            f"You are evaluating a job interview answer.\n"
            f"Question ({category}): {question}\n\n"
            "Listen carefully to the audio answer. "
            "Assess technical accuracy (0-100), communication clarity (0-100), "
            "confidence from vocal tone and pacing (0-100), and overall quality (0-100). "
            "Reply with ONLY a JSON object: "
            '{"technical_accuracy":0,"communication_clarity":0,"confidence":0,"overall_score":0,"brief_feedback":""}'
        )

        response = client.chat.complete(
            model=AUDIO_CHAT_MODEL,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "input_audio", "input_audio": audio_b64},
                    {"type": "text",        "text": prompt},
                ],
            }],
            temperature=0.2,
            max_tokens=512,
        )

        raw = response.choices[0].message.content.strip()
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]

        audio_eval = json.loads(raw)
        # Merge with a note that this came from audio analysis
        audio_eval["_source"] = "voxtral-audio"
        return audio_eval

    except Exception as e:
        print(f"  (Audio evaluation failed: {e} — falling back to text evaluation)")
        return evaluate_answer_text(
            question, category, transcript, RESUME_CONTEXT
        )


print("Answer evaluation functions ready.")

Answer evaluation functions ready.


---
## Cell 11 — Interview State

Initialises the interview session state.  
All answers, transcripts, and scores are accumulated here.

In [ ]:
# Interview session state
interview_session = {
    "candidate_name": name,
    "target_role":    TARGET_ROLE,
    "target_company": TARGET_COMPANY,
    "questions":      QUESTIONS,
    "responses": [],   # filled during the loop
    # Each response: {question, category, transcript, wav_path, evaluation}
}

print(f"Interview session initialised.")
print(f"Candidate   : {interview_session['candidate_name']}")
print(f"Role        : {interview_session['target_role']}")
print(f"Company     : {interview_session['target_company']}")
print(f"Questions   : {len(interview_session['questions'])}")

Interview session initialised.
Candidate   : John Doe
Role        : Senior Software Engineer
Company     : a top-tier tech company
Questions   : 7


---
## 🔊 PRE-INTERVIEW TTS TEST

**Run this cell BEFORE starting the interview to verify TTS is working.**  
You should hear the interviewer introduce themselves.

In [ ]:
# Pre-interview TTS verification
print("\n" + "="*60)
print("PRE-INTERVIEW AUDIO TEST")
print("="*60)
print("\nYou should hear the following audio:")
print("Make sure your volume is UP and speakers/headphones are ON.\n")

test_message = (
    f"Hello, {name}. This is your pre-interview audio check. "
    "If you can hear this clearly, your text to speech is working correctly. "
    "You are interviewing for the position of " + TARGET_ROLE + ". "
    "Let's begin."
)

speak(test_message, label="PRE-INTERVIEW TEST")

print("\n" + "="*60)
print("✓ If you heard that, proceed to Cell 12 to start the interview.")
print("✗ If you did NOT hear it:")
print("  1. Check Windows volume (taskbar, bottom-right)")
print("  2. Check Volume Mixer - ensure Python/Jupyter is NOT muted")
print("  3. Verify correct audio output device")
print("  4. Try running the TTS diagnostic cell above to test different voices")
print("="*60)


PRE-INTERVIEW AUDIO TEST

You should hear the following audio:
Make sure your volume is UP and speakers/headphones are ON.


[PRE-INTERVIEW TEST] Hello, John Doe. This is your pre-interview audio check. If you can hear this clearly, your text to speech is working correctly. You are interviewing for the position of Senior Software Engineer. Let's begin.

✓ If you heard that, proceed to Cell 12 to start the interview.
✗ If you did NOT hear it:
  1. Check Windows volume (taskbar, bottom-right)
  2. Check Volume Mixer - ensure Python/Jupyter is NOT muted
  3. Verify correct audio output device
  4. Try running the TTS diagnostic cell above to test different voices


---
## Cell 12 — Main Interview Loop

**⚠️ IMPORTANT: Run the PRE-INTERVIEW TEST cell above FIRST to verify TTS works!**

This is the core of the simulator. For each question it:

1. Speaks the question aloud via TTS
2. Records the candidate's spoken answer from the microphone
3. Plays back the recording for confirmation
4. Transcribes using Voxtral STT
5. Evaluates the answer (text + audio-native)
6. Speaks brief feedback via TTS
7. Stores everything in `interview_session`

> **Run this cell to start the interview.**  
> The cell will block and wait for your voice after each question.

In [ ]:
import time

# ── Opening statement
opening = (
    f"Welcome, {name}. My name is Alex and I will be conducting your interview today "
    f"for the position of {TARGET_ROLE} at {TARGET_COMPANY}. "
    "We have a few questions for you. Please answer as clearly and thoroughly as you can. "
    "The microphone will start recording automatically after each question. Let's begin."
)
speak(opening, label="INTERVIEWER")
time.sleep(1)

# ── Interview loop
for idx, q_item in enumerate(QUESTIONS, start=1):
    question = q_item["question"]
    category = q_item["category"]

    print(f"\n{'='*60}")
    print(f"Question {idx}/{len(QUESTIONS)}  [{category}]")
    print(f"{'='*60}")

    # 1. Speak the question
    speak(question, label="INTERVIEWER")
    time.sleep(0.5)

    # 2. Record answer
    try:
        wav_path = record_answer(duration_seconds=60)
    except Exception as e:
        print(f"  Recording failed: {e} — skipping question.")
        continue

    # 3. Playback for confirmation
    playback_audio(wav_path)

    # 4. Transcribe with Voxtral
    try:
        transcript = transcribe_audio(wav_path, context_hints=STT_CONTEXT_HINTS)
    except Exception as e:
        transcript = "[transcription failed]"
        print(f"  STT error: {e}")

    if not transcript or transcript == "[transcription failed]":
        speak("I didn't catch that. Let's move to the next question.", label="INTERVIEWER")
        continue

    # 5. Evaluate answer (text path — always reliable)
    print("  Evaluating...")
    try:
        evaluation = evaluate_answer_text(
            question, category, transcript, RESUME_CONTEXT
        )
    except Exception as e:
        print(f"  Evaluation error: {e}")
        evaluation = {
            "overall_score": 0,
            "brief_feedback": "I was unable to evaluate that answer.",
            "strengths": [],
            "improvements": [],
        }

    score = evaluation.get("overall_score", 0)
    feedback = evaluation.get("brief_feedback", "")

    print(f"  Score: {score}/100")
    print(f"  Feedback: {feedback}")

    # 6. Speak brief feedback
    transition = f"Thank you. {feedback}"
    if idx < len(QUESTIONS):
        transition += " Let's move on to the next question."
    speak(transition, label="INTERVIEWER")
    time.sleep(0.5)

    # 7. Store response
    interview_session["responses"].append({
        "question_number": idx,
        "category":        category,
        "question":        question,
        "transcript":      transcript,
        "wav_path":        wav_path,
        "evaluation":      evaluation,
    })

# ── Closing statement
speak(
    "That concludes the interview. Thank you for your time today. "
    "We will now generate your performance report.",
    label="INTERVIEWER"
)
print("\nInterview complete. Proceed to the next cell for the final report.")


[INTERVIEWER] Welcome, John Doe. My name is Alex and I will be conducting your interview today for the position of Senior Software Engineer at a top-tier tech company. We have a few questions for you. Please answer as clearly and thoroughly as you can. The microphone will start recording automatically after each question. Let's begin.

Question 1/7  [Technical]

[INTERVIEWER] At ABC Corporation, you used Apache Spark and AWS Glue to design data pipelines. Can you walk me through a scenario where you had to optimize a Spark job that was running slower than expected? What specific techniques or configurations did you use to improve its performance?

  Recording... (speak now, max 60s, auto-stops after 3.0s silence)
  Press Ctrl+C to stop early.
  (silence detected — stopping)
  Recorded 3.5s → interview_answer_ff5_bnqj.wav
  Playback:


  Transcribing with Voxtral...
  STT error: 1 validation error for AudioTranscriptionRequest
context_bias
  Input should be a valid list [type=list_type, input_value='Machine Learning,Data An...tation,Churn Prediction', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type

[INTERVIEWER] I didn't catch that. Let's move to the next question.

Question 2/7  [Technical]

[INTERVIEWER] You’ve listed both Python and R as languages you’ve used for machine learning. In your experience, what are the key trade-offs between using Python’s scikit-learn and R’s caret or tidymodels for building predictive models? Can you give an example where you chose one over the other and why?

  Recording... (speak now, max 60s, auto-stops after 3.0s silence)
  Press Ctrl+C to stop early.
  (silence detected — stopping)
  Recorded 3.5s → interview_answer_4xur7oi0.wav
  Playback:


  Transcribing with Voxtral...
  STT error: 1 validation error for AudioTranscriptionRequest
context_bias
  Input should be a valid list [type=list_type, input_value='Machine Learning,Data An...tation,Churn Prediction', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type

[INTERVIEWER] I didn't catch that. Let's move to the next question.

Question 3/7  [Behavioral]

[INTERVIEWER] At DEF Startups, you mentioned collaborating with data engineers to design data pipelines. Tell me about a time when there was a misalignment between your team and the data engineers on a project. How did you identify the issue, and what steps did you take to resolve it using the STAR method?

  Recording... (speak now, max 60s, auto-stops after 3.0s silence)
  Press Ctrl+C to stop early.
  (silence detected — stopping)
  Recorded 3.5s → interview_answer_vtwvdawt.wav
  Playback:


  Transcribing with Voxtral...
  STT error: 1 validation error for AudioTranscriptionRequest
context_bias
  Input should be a valid list [type=list_type, input_value='Machine Learning,Data An...tation,Churn Prediction', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type

[INTERVIEWER] I didn't catch that. Let's move to the next question.

Question 4/7  [Behavioral]

[INTERVIEWER] You led a team of data scientists at ABC Corporation to develop predictive models that increased sales by 25%. Describe a situation where a team member was struggling to meet their deliverables. How did you support them, and what was the outcome?

  Recording... (speak now, max 60s, auto-stops after 3.0s silence)
  Press Ctrl+C to stop early.
  (silence detected — stopping)
  Recorded 3.5s → interview_answer_bp4tx502.wav
  Playback:


  Transcribing with Voxtral...
  STT error: 1 validation error for AudioTranscriptionRequest
context_bias
  Input should be a valid list [type=list_type, input_value='Machine Learning,Data An...tation,Churn Prediction', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type

[INTERVIEWER] I didn't catch that. Let's move to the next question.

Question 5/7  [Project deep dive]

[INTERVIEWER] In your churn prediction project, you achieved a 20% reduction in customer churn. Can you dive deep into the model you built? Specifically, what features did you engineer, what algorithms did you evaluate, and how did you validate the model’s performance before deployment?

  Recording... (speak now, max 60s, auto-stops after 3.0s silence)
  Press Ctrl+C to stop early.
  (silence detected — stopping)
  Recorded 3.5s → interview_answer_y86bgpmg.wav
  Playback:


  Transcribing with Voxtral...
  STT error: 1 validation error for AudioTranscriptionRequest
context_bias
  Input should be a valid list [type=list_type, input_value='Machine Learning,Data An...tation,Churn Prediction', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type

[INTERVIEWER] I didn't catch that. Let's move to the next question.

Question 6/7  [Project deep dive]

[INTERVIEWER] Your customer segmentation project used clustering algorithms and resulted in a 25% increase in targeted marketing campaigns. How did you determine the optimal number of clusters, and what business metrics did you use to measure the success of the segmentation?

  Recording... (speak now, max 60s, auto-stops after 3.0s silence)
  Press Ctrl+C to stop early.
  (silence detected — stopping)
  Recorded 3.5s → interview_answer_dnrk9joh.wav
  Playback:


  Transcribing with Voxtral...
  STT error: 1 validation error for AudioTranscriptionRequest
context_bias
  Input should be a valid list [type=list_type, input_value='Machine Learning,Data An...tation,Churn Prediction', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type

[INTERVIEWER] I didn't catch that. Let's move to the next question.

Question 7/7  [Problem solving]

[INTERVIEWER] Imagine you’re joining our team as a Senior Software Engineer, and you’re tasked with improving the latency of a real-time recommendation system that currently processes 10,000 requests per second. The system uses a pre-trained machine learning model, but the response time has degraded to 500ms. Given your experience with Apache Spark and data pipelines, how would you approach diagnosing and resolving this issue?

  Recording... (speak now, max 60s, auto-stops after 3.0s silence)
  Press Ctrl+C to stop early.
  (silence detected — stopping)
  Recorded 3.5s → int

  Transcribing with Voxtral...
  STT error: 1 validation error for AudioTranscriptionRequest
context_bias
  Input should be a valid list [type=list_type, input_value='Machine Learning,Data An...tation,Churn Prediction', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type

[INTERVIEWER] I didn't catch that. Let's move to the next question.

[INTERVIEWER] That concludes the interview. Thank you for your time today. We will now generate your performance report.

Interview complete. Proceed to the next cell for the final report.


---
## Cell 13 — Final Interview Report

Uses `mistral-large-latest` to synthesise all individual question scores and  
transcripts into a comprehensive performance report.

In [ ]:
def generate_final_report(session: dict) -> dict:
    """
    Generate a comprehensive interview performance report.

    Returns a dict with:
        overall_score         : int (0-100)
        grade                 : str ("Excellent" / "Good" / "Average" / "Below Average")
        hire_recommendation   : str
        strengths             : list[str]
        weaknesses            : list[str]
        areas_to_improve      : list[str]
        preparation_topics    : list[str]
        question_breakdown    : list[{category, score, summary}]
        executive_summary     : str
    """
    responses = session.get("responses", [])
    if not responses:
        return {"error": "No responses recorded."}

    # Build Q&A summary for the prompt
    qa_summary = []
    scores = []
    for r in responses:
        ev = r.get("evaluation", {})
        overall = ev.get("overall_score", 0)
        scores.append(overall)
        qa_summary.append(
            f"Q{r['question_number']} [{r['category']}] (score: {overall}/100)\n"
            f"  Question  : {r['question']}\n"
            f"  Answer    : {r['transcript'][:400]}\n"
            f"  Strengths : {'; '.join(ev.get('strengths', []))}\n"
            f"  Improve   : {'; '.join(ev.get('improvements', []))}"
        )

    avg_score = round(sum(scores) / len(scores)) if scores else 0

    system_prompt = (
        "You are a senior interview assessor producing a final hiring recommendation report. "
        "Be honest, specific, and actionable. Base every point on evidence from the Q&A."
    )

    user_prompt = f"""Produce a final interview report for:

Candidate : {session['candidate_name']}
Role      : {session['target_role']}
Company   : {session['target_company']}
Avg Score : {avg_score}/100

=== QUESTION-BY-QUESTION PERFORMANCE ===
{chr(10).join(qa_summary)}
=========================================

Generate a comprehensive final report. Return ONLY valid JSON (no markdown fences):
{{
  "overall_score": {avg_score},
  "grade": "",
  "hire_recommendation": "",
  "strengths": [],
  "weaknesses": [],
  "areas_to_improve": [],
  "preparation_topics": [],
  "question_breakdown": [
    {{"question_number": 1, "category": "", "score": 0, "summary": ""}}
  ],
  "executive_summary": ""
}}"""

    response = client.chat.complete(
        model=EVALUATION_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.3,
        max_tokens=2048,
    )

    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    return json.loads(raw)


# ── Generate report
print("Generating final interview report...\n")
FINAL_REPORT = generate_final_report(interview_session)
interview_session["final_report"] = FINAL_REPORT

print("Report generated.")

Generating final interview report...

Report generated.


---
## Cell 14 — Display Final Report

Renders the report as a formatted printout and speaks the executive summary.

In [ ]:
from IPython.display import Markdown, display as ipy_display

def display_report(report: dict, session: dict) -> None:
    """Pretty-print the final interview report in the notebook."""

    if "error" in report:
        print(f"Report error: {report['error']}")
        return

    score = report.get("overall_score", 0)
    grade = report.get("grade", "N/A")
    hire  = report.get("hire_recommendation", "N/A")

    # Score colour
    if score >= 80:
        score_indicator = "STRONG"
    elif score >= 60:
        score_indicator = "GOOD"
    elif score >= 40:
        score_indicator = "AVERAGE"
    else:
        score_indicator = "BELOW AVERAGE"

    md_lines = [
        "# Interview Performance Report",
        "",
        f"**Candidate**   : {session['candidate_name']}  ",
        f"**Role**        : {session['target_role']}  ",
        f"**Company**     : {session['target_company']}  ",
        "",
        "---",
        "",
        f"## Overall Score: {score}/100 — {score_indicator} ({grade})",
        "",
        f"**Hiring Recommendation:** {hire}",
        "",
        "---",
        "",
        "## Executive Summary",
        "",
        report.get("executive_summary", ""),
        "",
        "---",
        "",
        "## Question Breakdown",
        "",
        "| # | Category | Score | Summary |",
        "|---|---|---|---|",
    ]

    for qb in report.get("question_breakdown", []):
        n   = qb.get("question_number", "?")
        cat = qb.get("category", "")
        sc  = qb.get("score", 0)
        sm  = qb.get("summary", "").replace("|", "/")
        md_lines.append(f"| {n} | {cat} | {sc}/100 | {sm} |")

    md_lines += [
        "",
        "---",
        "",
        "## Strengths",
        "",
    ]
    for s in report.get("strengths", []):
        md_lines.append(f"- {s}")

    md_lines += [
        "",
        "## Weaknesses",
        "",
    ]
    for w in report.get("weaknesses", []):
        md_lines.append(f"- {w}")

    md_lines += [
        "",
        "## Areas to Improve",
        "",
    ]
    for a in report.get("areas_to_improve", []):
        md_lines.append(f"- {a}")

    md_lines += [
        "",
        "## Suggested Preparation Topics",
        "",
    ]
    for t in report.get("preparation_topics", []):
        md_lines.append(f"- {t}")

    ipy_display(Markdown("\n".join(md_lines)))


display_report(FINAL_REPORT, interview_session)

# Speak the executive summary
summary = FINAL_REPORT.get("executive_summary", "")
score   = FINAL_REPORT.get("overall_score", 0)

spoken_report = (
    f"Here is your interview performance report, {name}. "
    f"Your overall score is {score} out of 100. "
    f"{summary} "
    "The full written report is displayed in the notebook above. Good luck with your job search."
)
speak(spoken_report, label="INTERVIEWER")

Report error: No responses recorded.

[INTERVIEWER] Here is your interview performance report, John Doe. Your overall score is 0 out of 100.  The full written report is displayed in the notebook above. Good luck with your job search.


---
## Cell 15 — Save Full Session to JSON

Persists the entire interview session — questions, transcripts, per-question evaluations,  
and the final report — to a timestamped JSON file in `outputs/interviews/`.

In [ ]:
import datetime

def save_session(session: dict, output_dir: Path) -> Path:
    """Save the full interview session to a JSON file."""
    output_dir.mkdir(parents=True, exist_ok=True)

    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_name = session["candidate_name"].replace(" ", "_").lower()
    filename  = f"interview_{safe_name}_{ts}.json"
    path      = output_dir / filename

    # Remove wav_path from saved output (local temp paths are not portable)
    serialisable = json.loads(json.dumps(session, default=str))
    for r in serialisable.get("responses", []):
        r.pop("wav_path", None)

    path.write_text(json.dumps(serialisable, indent=2, ensure_ascii=False), encoding="utf-8")
    return path


output_path = save_session(
    interview_session,
    output_dir=PROJECT_ROOT / "outputs" / "interviews",
)

print(f"Interview session saved to: {output_path}")

Interview session saved to: C:\Users\HP\Documents\Career_assitant\outputs\interviews\interview_john_doe_20260301_025007.json


---
## Cell 16 — (Optional) Replay a Single Question

Use this utility cell to replay any question without re-running the full interview.  
Useful for practising a specific question category.

In [ ]:
# ── Set the question index to replay (0-indexed)
REPLAY_INDEX = 0   # 0 = first question

if REPLAY_INDEX < len(QUESTIONS):
    q_item = QUESTIONS[REPLAY_INDEX]
    print(f"Replaying Q{REPLAY_INDEX + 1} [{q_item['category']}]")
    speak(q_item["question"], label="INTERVIEWER")

    wav = record_answer(duration_seconds=60)
    playback_audio(wav)

    transcript = transcribe_audio(wav, context_hints=STT_CONTEXT_HINTS)
    evaluation = evaluate_answer_text(
        q_item["question"], q_item["category"], transcript, RESUME_CONTEXT
    )

    print(f"\nScore     : {evaluation['overall_score']}/100")
    print(f"Feedback  : {evaluation['brief_feedback']}")
    print(f"Strengths : {evaluation.get('strengths', [])}")
    print(f"Improve   : {evaluation.get('improvements', [])}")

    speak(evaluation["brief_feedback"], label="FEEDBACK")
else:
    print(f"REPLAY_INDEX {REPLAY_INDEX} is out of range (0 – {len(QUESTIONS)-1}).")

Replaying Q1 [Technical]

[INTERVIEWER] At ABC Corporation, you used Apache Spark and AWS Glue to design data pipelines. Can you walk me through a scenario where you had to optimize a Spark job that was running slower than expected? What specific techniques or configurations did you use to improve its performance?

  Recording... (speak now, max 60s, auto-stops after 3.0s silence)
  Press Ctrl+C to stop early.
  (silence detected — stopping)
  Recorded 3.5s → interview_answer_f54wuzbz.wav
  Playback:


  Transcribing with Voxtral...


ValidationError: 1 validation error for AudioTranscriptionRequest
context_bias
  Input should be a valid list [type=list_type, input_value='Machine Learning,Data An...tation,Churn Prediction', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type

---
## Appendix — Model & API Reference

| Task | Model | API call |
|---|---|---|
| Question generation | `mistral-large-latest` | `client.chat.complete()` |
| TTS (interviewer voice) | `pyttsx3` (OS engine) | `engine.say()` — offline, free |
| STT (candidate answer) | `voxtral-mini-latest` | `client.audio.transcriptions.complete()` |
| Audio-native evaluation | `voxtral-small-latest` | `client.chat.complete()` with `input_audio` |
| Answer evaluation | `mistral-large-latest` | `client.chat.complete()` |
| Final report | `mistral-large-latest` | `client.chat.complete()` |

### Voxtral STT — Supported Audio Formats
MP3, WAV, FLAC, OGG, M4A, WEBM · max 3 hours · up to 2 GB

### Silence Detection Tuning
If the recorder stops too early (quiet environment) or too late (noisy),  
adjust `silence_threshold` and `silence_timeout` in `record_answer()`:

```python
wav = record_answer(
    duration_seconds=90,
    silence_threshold=0.005,   # raise if too sensitive
    silence_timeout=4.0,       # increase for slower speakers
)
```